# 4. Interaktiv Visualisering (EXTRA)
Plotly-baserade interaktiva plottar för presentationen.

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from pathlib import Path

_here = Path().resolve()
DATA_DIR = _here / 'data' if (_here / 'data').exists() else _here.parent / 'data'

df = pd.read_csv(DATA_DIR / 'cleaned_data.csv')
FEATURE_COLS = ['gdp', 'social_support', 'health', 'freedom', 'generosity', 'corruption', 'dystopia_residual']
df_latest = df[df['Year'] == df['Year'].max()].copy()

X = df_latest[FEATURE_COLS].values
X_scaled = StandardScaler().fit_transform(X)

pca2 = PCA(n_components=2)
X_pca = pca2.fit_transform(X_scaled)
df_latest['PC1'] = X_pca[:, 0]
df_latest['PC2'] = X_pca[:, 1]

In [2]:
fig = px.scatter(
    df_latest,
    x='PC1', y='PC2',
    color='happiness_score',
    color_continuous_scale='RdYlGn',
    hover_name='country',
    hover_data={'happiness_score': ':.2f', 'Rank': True, 'PC1': False, 'PC2': False},
    labels={
        'PC1': f'PC1 ({pca2.explained_variance_ratio_[0]*100:.1f}%)',
        'PC2': f'PC2 ({pca2.explained_variance_ratio_[1]*100:.1f}%)',
        'happiness_score': 'Happiness'
    },
    title=f'Interaktiv PCA – WHR26 {df_latest["Year"].iloc[0]}'
)
fig.write_html(str(DATA_DIR / 'pca_interactive.html'))
fig.show()

In [3]:
fig = px.parallel_coordinates(
    df_latest,
    dimensions=FEATURE_COLS,
    color='happiness_score',
    color_continuous_scale='RdYlGn',
    title='Parallellkoordinater – bidrag per variabel'
)
fig.write_html(str(DATA_DIR / 'parallel_coords.html'))
fig.show()

In [4]:
fig = px.scatter_matrix(
    df_latest,
    dimensions=FEATURE_COLS,
    color='happiness_score',
    color_continuous_scale='RdYlGn',
    hover_name='country',
    title='Scatter-matrix – alla features'
)
fig.update_traces(diagonal_visible=False, marker=dict(size=3))
fig.write_html(str(DATA_DIR / 'scatter_matrix.html'))
fig.show()

In [5]:
fig = px.scatter(
    df.sort_values('Year'),
    x='gdp', y='happiness_score',
    animation_frame='Year',
    hover_name='country',
    color='happiness_score',
    color_continuous_scale='RdYlGn',
    size_max=8,
    range_x=[df['gdp'].min() - 0.1, df['gdp'].max() + 0.1],
    range_y=[df['happiness_score'].min() - 0.2, df['happiness_score'].max() + 0.2],
    labels={'gdp': 'Log GDP per capita', 'happiness_score': 'Happiness score'},
    title='Happiness vs BNP – animerat 2019–2025'
)
fig.write_html(str(DATA_DIR / 'animated_scatter.html'))
fig.show()